# Task 14 -- Cross-cell/cross-time consistency: build-or-defer scoping (`docs/open_questions.md` §8.2)

`open_questions.md` §8.2 is explicit this is structural, not a tolerance
to tune -- so this is a **feasibility/scoping study**: how often does real
data actually exhibit the kind of cross-cell/cross-time violation a joint
spatiotemporal check would need to catch, and what would the cheapest
version cost? See `tasks/14-spatiotemporal-consistency-scoping.md`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorly as tl
import xarray as xr
from tensorly.decomposition import partial_tucker

from train.ingest import load_training_window

tl.set_backend("numpy")

SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("../data")

# Window entirely inside era 3 (starts 2025-10-27, core.archive_eras.ARCHIVE_ERAS);
# load_training_window rejects any window spanning more than one era.
T_START = datetime(2025, 11, 1, tzinfo=timezone.utc)
T_END = datetime(2025, 11, 15, tzinfo=timezone.utc)

VARIABLE = "wsh"
X_raw = load_training_window(DATA_DIR, T_START, T_END, [VARIABLE])
X = X_raw[VARIABLE].values  # (T, n_lat, n_lon)
print(f"loaded {X.shape[0]} snapshots at 3h cadence over a storm-active window")


## Real frame-to-frame advection-speed statistic for the storm-peak cell

In [ ]:
# For each pair of consecutive snapshots, find the peak-wsh cell and its
# physical displacement -- a concrete, computable diagnostic of how much
# there actually is for a joint spatiotemporal check to catch.
lat_step_deg, lon_step_deg = 0.25, 0.25  # core.config.WAVE_GRID_LAT_STEP_DEG/GRID_LON_STEP_DEG
km_per_deg_lat = 111.0
dt_hours = 3.0

peak_displacements_km_per_h = []
for t in range(X.shape[0] - 1):
    peak_a = np.unravel_index(np.nanargmax(X[t]), X[t].shape)
    peak_b = np.unravel_index(np.nanargmax(X[t + 1]), X[t + 1].shape)
    d_lat = (peak_b[0] - peak_a[0]) * lat_step_deg
    d_lon = (peak_b[1] - peak_a[1]) * lon_step_deg
    lat_mid = X_raw["wave_lat"].values[peak_a[0]]
    km_per_deg_lon = km_per_deg_lat * np.cos(np.radians(lat_mid))
    dist_km = np.hypot(d_lat * km_per_deg_lat, d_lon * km_per_deg_lon)
    peak_displacements_km_per_h.append(dist_km / dt_hours)

peak_displacements_km_per_h = np.array(peak_displacements_km_per_h)
print(f"peak-cell frame-to-frame speed: median {np.median(peak_displacements_km_per_h):.1f} km/h, "
      f"p99 {np.percentile(peak_displacements_km_per_h, 99):.1f} km/h, "
      f"max {peak_displacements_km_per_h.max():.1f} km/h")
# Typical storm system translation speeds are on the order of 20-50 km/h;
# outliers well above that in real, physically valid data indicate the
# peak-tracking proxy itself is noisy (e.g. multiple local maxima), which is
# itself useful evidence for how noisy any single-neighborhood check would be.


## Scoping questions (record, don't resolve as a single parameter)

1. **How often does real data exhibit implausible advection** (per the
   statistic above, or a spatial-gradient-based one) at a rate that a
   per-point-independent check structurally cannot catch?
2. **Cheapest plausible joint check:** a local finite-difference advection-
   speed bound over a small spatial neighborhood + one adjacent timestep,
   extending `physical_consistency_constraints.md` §5's tolerance-
   calibration method to a 2-point neighborhood -- what would its
   false-positive rate be on real, physically valid data (using the
   distribution measured above as the calibration input)?
3. **Added cost** (state-tracking: needs the previous timestep's full field
   at check time, not just the current query point) **vs. current
   coverage** (the three existing per-point constraints already catch
   magnitude/PM/period implausibility at each point individually).

Output a **build-now / defer / defer-indefinitely recommendation** in
`tasks/14-spatiotemporal-consistency-scoping.md` §4 -- explicitly not a
calibrated `delta` parameter the way Task 12's is.